# Latent Channel Propagation — Phase 1 Pipeline (Colab orchestrator)

Runs the five Phase 1 data-collection scripts in order:
`01_build_corpus.py` → `02_build_prompt_templates.py` → `03_run_inference_and_cache.py` → `04_label_vea.py` → `05_partition_cells.py`

See `README.md` in this folder for what each script does. This notebook is a thin orchestrator — it does not reimplement pipeline logic, it just calls the scripts with the right arguments and shows you the intermediate outputs.

**Before running:** Runtime → Change runtime type → select a GPU (T4 is fine for the 8B pilot).

## 0. Get the code onto this Colab instance

Pick ONE of the two cells below depending on whether `latent-channel-propagation` is a git repo you've pushed somewhere, or you're just uploading the folder directly.

In [ ]:
# Option A -- clone from your own git remote (uncomment and edit)
!git clone https://github.com/manchitro/latent-channel-propagation latent-channel-propagation
%cd latent-channel-propagation

In [ ]:
# Option B -- mount Google Drive if you've synced the folder there,
# or use the Colab file-upload UI (folder icon in the left sidebar) to
# drag the 01..05 .py files + README.md into /content/latent-channel-propagation
# from google.colab import drive
# drive.mount('/content/drive')

# Example if synced under My Drive/latent-channel-propagation -- edit path as needed:
# %cd /content/drive/MyDrive/latent-channel-propagation

In [ ]:
import os
assert os.path.exists('01_build_corpus.py'), (
    "Not in the right directory -- cd into the folder containing the "
    "01..05 scripts before continuing (see Option A/B above)."
)
print('Working directory OK:', os.getcwd())
!ls

## 0b. Mount Google Drive

Needed for two things: (1) reading your saved Hugging Face token from `.env` so you don't log in every session, and (2) persisting pipeline outputs at the end (Section 9). Run this once per session even though the code itself was cloned via git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0c. Data directory (all pipeline files live on Drive)

Every corpus/template/transcript/activation file this notebook produces is now written straight to `DATA_DIR` on Drive instead of `/content`. That means a full runtime recycle (not just a brief disconnect) no longer forces you to redo Steps 1 and 2 -- each generation step below checks whether its output already exists on Drive and skips straight past it if so.

In [ ]:
import os

DATA_DIR = '/content/drive/MyDrive/latent-channel-propagation/data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'DATA_DIR = {DATA_DIR}')

## 1. Install dependencies

In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes huggingface_hub pandas pyarrow

## 2. Authenticate with Hugging Face

Needed for `01_build_corpus.py` (AdvBench + WMDP) and `03_run_inference_and_cache.py` (Llama-3-8B-Instruct is a gated model -- request access on its model page first).

**One-time setup (VS Code Colab extension -- Colab's Secrets panel isn't available here):**
1. Mount Drive (cell above).
2. Create a file at `/content/drive/MyDrive/latent-channel-propagation/.env` containing one line: `HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx`
3. Re-run this cell on every future session -- it reads the token from Drive automatically, no re-entry needed.

**Do not commit `.env` to git** -- it's already covered by `.gitignore` in this repo. If Drive isn't mounted or the file isn't found, this falls back to the interactive `notebook_login()` flow.

In [ ]:
import os
from huggingface_hub import login, notebook_login

!pip install -q python-dotenv
from dotenv import load_dotenv

candidate_env_paths = [
    '/content/drive/MyDrive/latent-channel-propagation/.env',
    '.env',
]
loaded = False
for p in candidate_env_paths:
    if os.path.exists(p):
        load_dotenv(p)
        loaded = True
        print(f'Loaded env file: {p}')
        break

hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print('Logged in via HF_TOKEN from .env.')
else:
    print('No HF_TOKEN found in .env (or Drive not mounted) -- falling back to interactive login.')
    notebook_login()

## 3. Configuration

`SMOKE_TEST=True` caps the pipeline to a handful of prompts so you can validate the full chain runs end-to-end before committing GPU time to the full corpus. Flip to `False` for the real pilot run.

In [ ]:
SMOKE_TEST = True
SMOKE_LIMIT = 6  # rows fed to 03_run_inference_and_cache.py when SMOKE_TEST

WMDP_CAP = 5 if SMOKE_TEST else 200  # per-subset cap, passed to 01_build_corpus.py

MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # pre-quantized, actively maintained -- see 03_run_inference_and_cache.py docstring. Swap for the 70B scale-up on AIUB hardware
USE_LLM_JUDGE = False  # flip once you've implemented llm_judge_label() in 04_label_vea.py

print(f"SMOKE_TEST={SMOKE_TEST}  WMDP_CAP={WMDP_CAP}  MODEL_ID={MODEL_ID}")

## 4. Step 1 -- Build corpus (AdvBench + WMDP)

Skips straight to loading if `corpus.parquet` already exists on Drive from a previous run.

In [ ]:
corpus_path = f'{DATA_DIR}/corpus.parquet'

if os.path.exists(corpus_path):
    print(f'Found existing corpus at {corpus_path} -- skipping rebuild.')
else:
    !python 01_build_corpus.py --out {corpus_path} --wmdp-per-subset-cap {WMDP_CAP}

In [ ]:
import pandas as pd
corpus = pd.read_parquet(corpus_path)
print(corpus.shape)
corpus.head()

## 5. Step 2 -- Build train/deploy prompt templates

Skips straight to loading if `templated_corpus.parquet` already exists on Drive.

In [ ]:
templated_path = f'{DATA_DIR}/templated_corpus.parquet'

if os.path.exists(templated_path):
    print(f'Found existing templated corpus at {templated_path} -- skipping rebuild.')
else:
    !python 02_build_prompt_templates.py --corpus {corpus_path} --out {templated_path}

In [ ]:
templated = pd.read_parquet(templated_path)
print(templated.shape)
templated[['variant_id', 'framing', 'user_prompt']].head()

## 6. Step 3 -- Run inference + cache activations

This is the GPU-heavy step, and the one Colab disconnects during. Transcripts, activations, and the row-level checkpoint all live on Drive now -- if you get disconnected (brief or full runtime recycle), just reconnect and re-run this exact cell; it skips everything already checkpointed and resumes from there.

In [ ]:
limit_flag = f"--limit {SMOKE_LIMIT}" if SMOKE_TEST else ""

transcripts_path = f'{DATA_DIR}/transcripts.parquet'
activations_path = f'{DATA_DIR}/activations.npz'
checkpoint_dir = f'{DATA_DIR}/checkpoint'

# HF cache also lives on Drive so the model is only ever downloaded once,
# not once per session -- this is separate from the row-level checkpoint
# above and matters even on a brief disconnect, since /content's default
# HF cache is wiped on any full runtime recycle.
hf_cache_dir = f'{DATA_DIR}/hf_cache'

!python 03_run_inference_and_cache.py \
    --templated {templated_path} \
    --model-id {MODEL_ID} \
    --four-bit \
    {limit_flag} \
    --checkpoint-dir {checkpoint_dir} \
    --cache-dir {hf_cache_dir} \
    --out-transcripts {transcripts_path} \
    --out-activations {activations_path}

In [ ]:
transcripts = pd.read_parquet(transcripts_path)
print(transcripts.shape)
print(transcripts['compliance_label'].value_counts())
transcripts[['variant_id', 'framing', 'compliance_label', 'answer']].head()

In [ ]:
import numpy as np
acts = np.load(activations_path)
print(f"{len(acts.files)} cached activation vectors")
sample_key = acts.files[0]
print(f"example '{sample_key}' shape: {acts[sample_key].shape}  (num_layers+1, hidden_dim)")

## 7. Step 4 -- Label verbalized evaluation awareness (VEA)

Skips straight to loading if `labeled_transcripts.parquet` already exists on Drive.

In [ ]:
labeled_path = f'{DATA_DIR}/labeled_transcripts.parquet'
judge_flag = "--use-llm-judge" if USE_LLM_JUDGE else ""

if os.path.exists(labeled_path):
    print(f'Found existing labeled transcripts at {labeled_path} -- skipping relabel.')
else:
    !python 04_label_vea.py \
        --transcripts {transcripts_path} \
        --out {labeled_path} \
        {judge_flag}

In [ ]:
labeled = pd.read_parquet(labeled_path)
print(labeled['vea_label'].value_counts())
labeled[['variant_id', 'framing', 'vea_label', 'reasoning']].head()

## 8. Step 5 -- Partition into the four analysis cells

Skips straight to loading if `partitioned_corpus.parquet` already exists on Drive.

In [ ]:
partitioned_path = f'{DATA_DIR}/partitioned_corpus.parquet'

if os.path.exists(partitioned_path):
    print(f'Found existing partitioned corpus at {partitioned_path} -- skipping repartition.')
else:
    !python 05_partition_cells.py --labeled {labeled_path} --out {partitioned_path}

In [ ]:
partitioned = pd.read_parquet(partitioned_path)
print(partitioned['cell'].value_counts())
partitioned.head()

## 9. Confirm outputs on Drive

Nothing to copy anymore -- every artifact was written directly to `DATA_DIR` on Drive as it was produced, so it already survives a runtime recycle. This cell just lists what's there so you can sanity-check before ending the session.

In [ ]:
for f in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, f)
    if os.path.isfile(full):
        print(f'{f}  ({os.path.getsize(full)/1024:.1f} KB)')
    else:
        print(f'{f}/')

## Next steps

- If `SMOKE_TEST` passed and the compliance/VEA labels look reasonable, delete the generated files under `DATA_DIR` (they were built from the capped smoke-test corpus) and set `SMOKE_TEST=False` before rerunning from Step 1 for the full pilot corpus.
- Once you're happy with the 8B pilot results, swap `MODEL_ID` to the 70B checkpoint and move this notebook (or the equivalent scripts) to the AIUB DGX Spark lab for the scale-up run.
- `partitioned_corpus.parquet` and `activations.npz` (both under `DATA_DIR`) are the inputs to Phase 2 (probe training) -- see `Latent Channel Propagation - Feasibility and Impact.md` in the Obsidian vault for the full plan.